# Step 4: Add Metadata and Analyze Spatial Patterns


This notebook turns metric results into spatial summaries. You will use PGA and FAS as two separate examples, so you can see how the same spatial tests can highlight different model-performance patterns for different metrics.


## Imports

These functions prepare metric fields and calculate spatial summaries.


In [ ]:
from pathlib import Path
import sys

# Prefer the source checkout that contains this notebook when running without an installed wheel.
repo_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "spatial_vtk").exists()
    ),
    Path.cwd(),
)
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from spatial_vtk.config.notebook import (
    notebook_timer,
    notebook_figure_sidecar_settings,
    prepare_notebook_geospatial_environment,
    register_svtk_cell_timer,
)
prepare_notebook_geospatial_environment()

with notebook_timer():
    import os

    os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

    from pathlib import Path
    import pandas as pd
    from IPython.display import Markdown, display

    from spatial_vtk.config import SpatialVTKConfig
    from spatial_vtk.config.labels import metric_display_name
    from spatial_vtk.io import read_config_table
    from spatial_vtk.spatial.calculate import run_spatial_statistics_workflow
    from spatial_vtk.spatial.map import plot_pca_summary, plot_residual_grid, plot_station_bias_map
    from spatial_vtk.spatial.plot import plot_distance_correlation_by_metric, plot_geology_contrast
    register_svtk_cell_timer()


## Configuration

Load the config and read the spatial-statistics settings from the tutorial run scenario.


In [ ]:
import os

from spatial_vtk.config.notebook import find_repo_root, notebook_figure_sidecar_settings

# Use the repository root so paths match the public source checkout.
repo_root = find_repo_root()
config_path = repo_root / "data/examples/configuration/example_spatial_vtk_config.yaml"

# Load the tutorial run scenario and make it the active config for later package calls.
cfg = SpatialVTKConfig.from_file(config_path, run_scenario="tutorial").activate()

# Step 4 uses a larger QC-passed metrics snapshot so per-metric spatial tests have enough stations.
metric_source_path = cfg.path("paths.metric_figure_snapshot")
figure_dir = cfg.path("outputs.figures")
figure_dir.mkdir(parents=True, exist_ok=True)
spatial_sidecars = notebook_figure_sidecar_settings("spatial", figure_dir=figure_dir)
spatial_metrics = ("PGA", "FAS")
add_basemap = os.environ.get("SVTK_ADD_BASEMAP", "0") == "1"


## Prepare Metric-Specific Spatial Fields

Spatial statistics use one value per event-station observation, plus station and event coordinates. This notebook delegates the table-building work to the package workflow, then uses the returned tables for compact displays and figures.

In [ ]:
# Read the configured site/geology metadata table and run the package spatial workflow.
site_metadata = read_config_table("paths.site_metadata")
spatial_result = run_spatial_statistics_workflow(
    metric_source_path,
    cfg=cfg,
    metric=spatial_metrics,
    station_metadata=site_metadata,
    verbose=True,
)

if spatial_result.failures:
    display(Markdown("### Non-fatal workflow diagnostics"))
    display(pd.DataFrame(spatial_result.failures))

# Keep small per-metric table handles for the plotting cells below.
metric_field = spatial_result.tables["metric_field"]
event_centered_residuals = spatial_result.tables["event_centered_residuals"]
station_bias = spatial_result.tables["station_bias"]
spatial_products = {}
for metric_name in spatial_result.metrics:
    display(Markdown(f"### {metric_display_name(metric_name)}"))
    field = metric_field.loc[metric_field["metric"].astype(str).eq(metric_name)].copy()
    centered = event_centered_residuals.loc[event_centered_residuals["metric"].astype(str).eq(metric_name)].copy()
    bias = station_bias.loc[station_bias["metric"].astype(str).eq(metric_name)].copy()
    spatial_products[metric_name] = {"field": field, "centered": centered, "station_bias": bias}
    display(
        pd.DataFrame(
            {
                "Output": ["Metric field", "Event-centered field", "Station bias"],
                "Rows": [len(field), len(centered), len(bias)],
                "Events": [field["event_id"].nunique() if "event_id" in field else "", centered["event_id"].nunique() if "event_id" in centered else "", ""],
                "Stations": [field["station"].nunique() if "station" in field else "", centered["station"].nunique() if "station" in centered else "", bias["station"].nunique() if "station" in bias else ""],
            }
        )
    )
    display(bias.head())


## Station Bias Maps

These maps show the mean event-centered residual at each station. Positive values mean the observed amplitudes are larger than the synthetic amplitudes on average for that metric.


In [ ]:
for metric_name, products in spatial_products.items():
    display(Markdown(f"### {metric_display_name(metric_name)}"))

    # Map the mean event-centered residual at each station for this metric.
    station_bias_fig = plot_station_bias_map(
        products["station_bias"],
        title=f"{metric_display_name(metric_name)} Station Bias",
        value_col="mean_centered",
        value_label="Mean event-centered log2(obs/syn)",
        add_basemap=add_basemap,
        showfig=True,
        savefig=True,
        outpath=figure_dir / f"step_04_{metric_name.lower()}_station_bias.png",
        **spatial_sidecars.kwargs(),
    )


## Residual Grid Maps

A residual grid gives you a quick spatial overview of where residuals are broadly positive or negative. These examples use the same event-centered residual field as the station-bias maps.


In [ ]:
for metric_name, products in spatial_products.items():
    display(Markdown(f"### {metric_display_name(metric_name)}"))

    # Grid event-centered residuals onto lon/lat cells for a broader spatial view.
    residual_grid_fig = plot_residual_grid(
        products["centered"],
        lon_col="lon",
        lat_col="lat",
        value_col="field_centered",
        cell_size_deg=0.05,
        title=f"{metric_display_name(metric_name)} Residual Grid",
        add_basemap=add_basemap,
        showfig=True,
        savefig=True,
        outpath=figure_dir / f"step_04_{metric_name.lower()}_residual_grid.png",
        **spatial_sidecars.kwargs(),
    )


## Spatial Correlation Tests

Moran's I and distance-bin correlations help you see whether residuals cluster in space. The workflow has already written these tables; this cell displays them and plots the distance-binned correlations together.

In [ ]:
# Use the Moran's I and distance-bin correlation tables written by the spatial workflow.
morans_i = spatial_result.tables["morans_i"]
distance_bins = spatial_result.tables["distance_bin_correlations"]

for metric_name in spatial_result.metrics:
    display(Markdown(f"### {metric_display_name(metric_name)}"))
    display(morans_i.loc[morans_i["metric"].astype(str).eq(metric_name)])
    display(distance_bins.loc[distance_bins["metric"].astype(str).eq(metric_name)].head())

# Plot correlation as a function of station separation distance for the selected metrics.
correlation_distance_fig = plot_distance_correlation_by_metric(
    distance_bins,
    significance_df=morans_i,
    title="Spatial Correlation by Distance",
    showfig=True,
    savefig=True,
    outpath=figure_dir / "step_04_spatial_correlation_distance.png",
    **spatial_sidecars.kwargs(),
)


## Clustering and PCA Spatial Modes

These summaries group stations with similar residual fingerprints and identify dominant station-level patterns. The workflow writes the clustering and PCA tables; this cell renders PCA summary figures from those outputs.

In [ ]:
# Use the clustering and PCA tables written by the spatial workflow.
clusters = spatial_result.tables["clusters"]
cluster_scores = spatial_result.tables["cluster_scores"]
cluster_summary = spatial_result.tables["cluster_summary"]
pca_station_scores = spatial_result.tables["pca_station_scores"]
pca_feature_loadings = spatial_result.tables["pca_feature_loadings"]
pca_explained_variance = spatial_result.tables["pca_explained_variance"]

for metric_name in spatial_result.metrics:
    display(Markdown(f"### {metric_display_name(metric_name)}"))
    station_scores = pca_station_scores.loc[pca_station_scores["metric"].astype(str).eq(metric_name)].copy()
    explained = pca_explained_variance.loc[pca_explained_variance["metric"].astype(str).eq(metric_name)].copy()
    loadings = pca_feature_loadings.loc[pca_feature_loadings["metric"].astype(str).eq(metric_name)].copy()

    # Plot the PC1 map, explained variance, and feature loading summary for this metric.
    pca_summary_fig = plot_pca_summary(
        station_scores,
        explained,
        loadings,
        mode="PC1",
        title=f"{metric_display_name(metric_name)} PCA Spatial Mode Summary",
        add_basemap=add_basemap,
        showfig=True,
        savefig=True,
        outpath=figure_dir / f"step_04_{metric_name.lower()}_pca_summary.png",
        **spatial_sidecars.kwargs(),
    )
    display(explained)


## Geology Contrasts

Compare PGA and FAS residuals across the configured geology classes. GeoJSON regions and corridors are handled in the next notebook.

In [ ]:
# Use the geology contrast table written by the spatial workflow.
geology_contrasts = spatial_result.tables["geology_contrasts"]

for metric_name, products in spatial_products.items():
    display(Markdown(f"### {metric_display_name(metric_name)}"))
    geology_contrast = geology_contrasts.loc[geology_contrasts["metric"].astype(str).eq(metric_name)].copy()

    # Plot the residual distributions and annotate the configured geology-class contrast for this metric.
    geology_contrast_fig = plot_geology_contrast(
        products["centered"],
        station_metadata=site_metadata,
        contrast_df=geology_contrast,
        title=f"{metric_display_name(metric_name)} Residuals by Geology Class",
        showfig=True,
        savefig=True,
        outpath=figure_dir / f"step_04_{metric_name.lower()}_geology_contrast.png",
        **spatial_sidecars.kwargs(),
    )
    display(geology_contrast)
